<a href="https://colab.research.google.com/github/vzyhug/200123035_14DHTH07_DL/blob/main/CNN/CNN_multiclass_classification_prj1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IMPORT LIB

In [1]:
import os
import pandas as pd
from glob import glob
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import json

# Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ĐƯỜNG DẪN TRÊN COLAB
DATA_ROOT = '/content/drive/MyDrive/test'
save_dir = '/content/drive/MyDrive/test/SAVE'
os.makedirs(save_dir, exist_ok=True)

# Lấy danh sách ảnh
image_paths = []
for ext in ['*.png', '*.jpg', '*.jpeg']:
    image_paths.extend(glob(os.path.join(DATA_ROOT, '**', ext), recursive=True))

# Tạo danh sách dữ liệu
records = []
for full_path in image_paths:
    rel_path = os.path.relpath(full_path, DATA_ROOT)
    logo_id = os.path.basename(os.path.dirname(full_path)) # Giả sử tên thư mục chứa ảnh là tên logo
    records.append({'filename': rel_path, 'logo_id': logo_id})

df = pd.DataFrame(records)

# Tạo mapping từ tên logo sang ID (số nguyên)
unique_logos = df['logo_id'].unique()
NUM_CLASSES = len(unique_logos)
label_to_id = {logo: int(i) for i, logo in enumerate(unique_logos)}
id_to_label = {int(i): logo for i, logo in enumerate(unique_logos)}

df['label'] = df['logo_id'].map(label_to_id)

# Lưu từ điển ID-Label lại để sau này dự đoán còn biết tên logo là gì
with open(os.path.join(save_dir, 'label_map.json'), 'w') as f:
    json.dump(id_to_label, f)

print(f"✅ Đã tìm thấy {len(df)} ảnh.")
print(f"✅ Tổng số loại logo (số lớp) cần phân loại: {NUM_CLASSES}")

Mounted at /content/drive


KeyError: 'logo_id'

## KHỞI TẠO DATASET

In [ ]:
def load_safe_image(img_path):
    try:
        img = Image.open(img_path)
        if img.mode in ('RGBA', 'LA', 'P'):
            img = img.convert('RGBA')
            background = Image.new('RGBA', img.size, (255, 255, 255))
            background.paste(img, mask=img)
            return background.convert('RGB')
        else:
            return img.convert('RGB')
    except Exception as e:
        return Image.new('RGB', (224, 224), (255, 255, 255))

class LogoClassificationDataset(Dataset):
    def __init__(self, dataframe, raw_dir, transform=None):
        self.dataframe = dataframe
        self.raw_dir = raw_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.raw_dir, row['filename'])
        label = row['label']

        img = load_safe_image(img_path)
        if self.transform:
            img = self.transform(img)

        return img, torch.tensor(label, dtype=torch.long)

## ĐỊNH NGHĨA MẠNG CNN

In [ ]:
class BasicLogoCNN(nn.Module):
    def __init__(self, num_classes):
        super(BasicLogoCNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 14 * 14, 512),
            nn.ReLU(),
            nn.Dropout(0.4), # Ngăn chặn học vẹt (overfitting)
            nn.Linear(512, num_classes) # Output số lượng nơ-ron bằng số lớp
        )

    def forward(self, x):
        x = self.conv_layers(x)
        logits = self.fc_layers(x)
        return logits

## TRAIN

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Đang chạy trên: {DEVICE}")

# Transforms: Tăng cường dữ liệu cho tập Train
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Chia dữ liệu Train (80%) và Val (20%)
full_dataset = LogoClassificationDataset(df, DATA_ROOT, transform=train_transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = torch.utils.data.random_split(full_dataset, [train_size, val_size])

# Sửa lại transform cho tập Val để không bị xoay ảnh ngẫu nhiên
val_data.dataset.transform = val_transform

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)

# Khởi tạo Model, Loss, Optimizer
model = BasicLogoCNN(num_classes=NUM_CLASSES).to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

EPOCHS = 20
best_acc = 0.0

for epoch in range(EPOCHS):
    # --- TRAIN ---
    model.train()
    train_loss, train_correct = 0, 0
    for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        train_correct += (preds == labels).sum().item()

    train_acc = train_correct / len(train_data)

    # --- VALIDATION ---
    model.eval()
    val_loss, val_correct = 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == labels).sum().item()

    val_acc = val_correct / len(val_data)

    print(f"Epoch {epoch+1} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

    # Lưu model nếu Accuracy trên tập Validation tốt hơn
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), os.path.join(save_dir, 'basic_cnn_classifier.pth'))
        print("⭐ Đã lưu model tốt nhất!")

## TEST

In [ ]:
import matplotlib.pyplot.pyplot as plt

def predict_logo(image_path, model, device, label_map):
    model.eval()

    # Mở và xử lý ảnh
    img = load_safe_image(image_path)
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = transform(img).unsqueeze(0).to(device)

    # Dự đoán
    with torch.no_grad():
        outputs = model(img_tensor)
        # Chuyển logits thành xác suất (percentages)
        probabilities = torch.nn.functional.softmax(outputs[0], dim=0)

        # Lấy nhãn có xác suất cao nhất
        max_prob, predicted_id = torch.max(probabilities, dim=0)

    predicted_label = label_map[str(predicted_id.item())]
    confidence = max_prob.item() * 100

    # Hiển thị kết quả
    plt.imshow(img)
    plt.axis('off')
    plt.title(f"Dự đoán: {predicted_label} ({confidence:.2f}%)")
    plt.show()

# Test thử với một ảnh trong dataset
import random
random_test_img = random.choice(image_paths)

# Đọc lại file mapping nhãn đã lưu
with open(os.path.join(save_dir, 'label_map.json'), 'r') as f:
    label_map = json.load(f)

# Load lại mô hình tốt nhất
model = BasicLogoCNN(num_classes=NUM_CLASSES).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(save_dir, 'basic_cnn_classifier.pth')))

predict_logo(random_test_img, model, DEVICE, label_map)